In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist
import pyranges as pr


In [23]:
calls = pd.read_pickle("./data/calls_loose.pkl")
calls

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,...,df_loco,phi,pval,p_loco,qval,neighbor_support,dominance_blocked,reason,category,call_reason
85,1,77801,77900,283.205748,16,0.509852,16,0,0.258787,16,...,15.0,1.536786,0.000000e+00,1.479381e-03,0.000000e+00,0,False,no_neighbor_support,euc_gene,rescue_isolated
125,1,117201,117300,209.579299,16,0.469321,16,0,0.315957,16,...,15.0,1.536786,0.000000e+00,9.155010e-12,0.000000e+00,1,False,ok,euc_gene,main
129,1,121701,121800,121.351245,16,0.345870,4,16,0.321142,9,...,15.0,1.536786,2.556039e-10,7.882232e-08,1.349449e-08,1,False,ok,euc_gene,main
142,1,123101,123200,60.386475,16,0.280947,0,9,0.235865,9,...,15.0,1.536786,9.862091e-04,2.243899e-02,1.093957e-02,1,False,ok,euc_gene,main
294,1,243401,243500,69.454337,16,0.201690,4,9,0.186608,0,...,15.0,1.536786,1.295724e-04,1.617305e-02,1.967793e-03,2,False,weak_effect,euc_gene,main
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304611,5,15242801,15242900,77.878127,16,0.302944,13,12,0.154973,12,...,15.0,1.544736,1.968215e-05,3.571681e-02,1.171208e-03,2,False,weak_effect,het_te,main
304613,5,15243001,15243100,86.855885,16,0.244768,8,9,0.176046,12,...,15.0,1.544736,2.232159e-06,2.235987e-03,1.974125e-04,2,False,weak_effect,het_te,main
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,...,15.0,1.544736,1.047749e-04,9.935832e-04,4.410364e-03,1,False,ok,het_te,main
304628,5,15259801,15259900,67.105317,16,0.465839,5,4,0.305060,4,...,15.0,1.544736,2.397276e-04,1.577884e-02,8.303919e-03,2,False,ok,het_te,main


In [24]:
calls[calls['start']==76901]

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,...,df_loco,phi,pval,p_loco,qval,neighbor_support,dominance_blocked,reason,category,call_reason
156870,1,76901,77000,55.496179,15,0.275298,2,4,0.211341,4,...,14.0,1.527399,0.001582,0.011624,0.024342,1,False,ok,euc_te,main


In [25]:
calls['chr']

85        1
125       1
129       1
142       1
294       1
         ..
304611    5
304613    5
304627    5
304628    5
304638    5
Name: chr, Length: 8116, dtype: int64

In [26]:
# pip install pyranges
import pandas as pd
import pyranges as pr

# --- Helpers ---
def chr_to_int(x):
    """Map 'chr1'/'1' -> 1; X->23, Y->24, M/MT->25; else None."""
    s = str(x).strip()
    s = s.replace("CHR", "chr").replace("Chr", "chr")
    if s.startswith("chr"):
        s = s[3:]
    if s in {"X", "x"}: return 23
    if s in {"Y", "y"}: return 24
    if s.upper() in {"M", "MT"}: return 25
    try:
        return int(s)
    except ValueError:
        return None

def read_gff_intervals_to_pr(gff_path, feature_type=None):
    """
    Read GFF/GFF3 to PyRanges with columns: Chromosome(int), Start, End.
    Converts 1-based inclusive -> 0-based half-open.
    If feature_type is given (e.g., 'gene' or 'scdmr'), keep only those rows.
    """
    rows = []
    with open(gff_path) as fh:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            chrom, source, ftype, start, end, score, strand, phase, attrs = line.rstrip("\n").split("\t")
            if feature_type and ftype.lower() != feature_type.lower():
                continue
            rows.append({
                "Chromosome": chr_to_int(chrom),
                "Start": int(start) - 1,   # 1-based inclusive -> 0-based half-open
                "End": int(end)
            })
    df = pd.DataFrame(rows).dropna(subset=["Chromosome"])
    df["Chromosome"] = df["Chromosome"].astype("int64")
    return pr.PyRanges(df)

def calls_df_to_pr(calls_df, convert_one_based=False):
    """
    Convert your calls df (columns: chr(int), start, end) to PyRanges.
    Set convert_one_based=True if calls are 1-based inclusive.
    """
    df = calls_df.copy()
    if convert_one_based:
        df["start"] = df["start"].astype(int) - 1
        # end stays as-is
    out = pd.DataFrame({
        "Chromosome": df["chr"].astype("int64"),
        "Start": df["start"].astype("int64"),
        "End": df["end"].astype("int64"),
    })
    return pr.PyRanges(out)

# --- Overlap utilities ---
def overlapping_calls(calls_df, gff_path, feature_type=None, min_overlap_bp=1, calls_are_one_based=False):
    """
    Return subset of calls (rows) that overlap any interval from the GFF.
    """
    pr_gff = read_gff_intervals_to_pr(gff_path, feature_type=feature_type)
    pr_calls = calls_df_to_pr(calls_df, convert_one_based=calls_are_one_based)

    joined = pr_calls.join(pr_gff, report_overlap=True)  # left join from calls to gff
    if "Overlap" in joined.columns:
        joined = joined[joined.Overlap >= min_overlap_bp]
    # deduplicate by original call coords
    subset = joined.df[["Chromosome", "Start", "End"]].drop_duplicates()

    # map back to your original schema
    out = subset.rename(columns={"Chromosome": "chr", "Start": "start", "End": "end"})
    return out.sort_values(["chr", "start", "end"]).reset_index(drop=True)

def gff_intervals_overlapping_calls(calls_df, gff_path, feature_type=None, min_overlap_bp=1, calls_are_one_based=False):
    """
    Return subset of GFF intervals that overlap any call.
    """
    pr_gff = read_gff_intervals_to_pr(gff_path, feature_type=feature_type)
    pr_calls = calls_df_to_pr(calls_df, convert_one_based=calls_are_one_based)

    joined = pr_gff.join(pr_calls, report_overlap=True)  # left join from gff to calls
    if "Overlap" in joined.columns:
        joined = joined[joined.Overlap >= min_overlap_bp]
    subset = joined.df[["Chromosome", "Start", "End"]].drop_duplicates()
    out = subset.rename(columns={"Chromosome": "chr", "Start": "start", "End": "end"})
    return out.sort_values(["chr", "start", "end"]).reset_index(drop=True)

# --- Example usage ---
# calls = pd.DataFrame({
#     "chr":  [1, 1, 1, 2],
#     "start":[76850, 77000, 90000, 1000],   # assume already 0-based half-open
#     "end":  [76950, 77100, 90100, 1100]
# })
# # If your GFF rows are scdmr features like:
# # chr1  scdmr  scdmr  76900  77300  1 . . .
# # you can filter with feature_type="scdmr" (or None to keep all entries).
#
# calls_subset = overlapping_calls(calls, "your.gff", feature_type="scdmr", min_overlap_bp=1)
# gff_subset   = gff_intervals_overlapping_calls(calls, "your.gff", feature_type="scdmr", min_overlap_bp=1)
# print("Calls overlapping GFF:\n", calls_subset)
# print("GFF intervals overlapping calls:\n", gff_subset)


In [27]:
calls_subset = overlapping_calls(calls, "/gale/raidix/rdx-7/jwalker/scdmr.gff")

calls_subset

,chr,start,end
0,1,76901,77000
1,1,77001,77100
2,1,77201,77300
3,1,281801,281900
4,1,281901,282000
...,...,...,...
3352,5,26885901,26886000
3353,5,26886001,26886100
3354,5,26886101,26886200
3355,5,26886201,26886300


In [28]:
gff_subset   = gff_intervals_overlapping_calls(calls, "/gale/raidix/rdx-7/jwalker/scdmr.gff")
gff_subset

,chr,start,end
0,1,76899,77300
1,1,281799,282000
2,1,282599,282800
3,1,394099,395300
4,1,704999,705200
...,...,...,...
1283,5,26552199,26552800
1284,5,26575699,26575900
1285,5,26866399,26867200
1286,5,26884999,26885700


In [29]:
calls_subset.to_pickle("./data/calls_filtered.pkl")

In [17]:
#sanity check
calls = pr.PyRanges(pd.DataFrame({
    "Chromosome":[1, 2],
    "Start":[0, 0],
    "End":[100, 100],
}))
gff = pr.PyRanges(pd.DataFrame({
    "Chromosome":[1],
    "Start":[0],
    "End":[100],
}))

# Only chr==1 overlaps; chr==2 does not
print(calls.join(gff).df)
# -> one row (Chromosome=1, Start=0, End=100), nothing from Chromosome=2


  Chromosome  Start  End  Start_b  End_b
0          1      0  100        0    100
